# Aula 7 — Explorando: agentes, grafos e deep agents
No modo fake, o modelo segue um **roteiro**. Leia o roteiro: é exatamente o que o modelo real faria.

In [ ]:
# Execute esta célula primeiro. Ela mostra se você está no modo OPENAI ou FAKE (a chave nunca aparece).
from assistente_ia.config import descrever_ambiente
print(descrever_ambiente())

In [ ]:
from langchain_core.messages import AIMessage
from assistente_ia.container import criar_agente_garcom

roteiro = [
    AIMessage(content="", tool_calls=[{"name": "consultar_preco", "args": {"produto": "bolo de rolo"}, "id": "1"}]),
    AIMessage(content="A fatia de bolo de rolo custa R$ 9,00."),
]
agente = criar_agente_garcom(roteiro)
print(agente.conversar("Quanto é o bolo de rolo?", "nb"))
for m in agente.historico("nb"):
    print(f"{m.type:>6}: {getattr(m, 'tool_calls', '') or m.content}")

## O desenho do grafo de atendimento
Cole a saída em https://mermaid.live para ver o diagrama.

In [ ]:
from assistente_ia.application.grafo_atendimento import criar_grafo_atendimento
from assistente_ia.container import criar_llm, criar_servico_rag

llm = criar_llm()
grafo = criar_grafo_atendimento(llm, criar_servico_rag("padaria", llm=llm))
print(grafo.get_graph().draw_mermaid())

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "nb-1"}}
pausa = grafo.invoke({"pergunta": "Quero encomendar 40 sonhos"}, config)
print("pausou:", pausa["__interrupt__"][0].value)
print(grafo.invoke(Command(resume="aprovar"), config)["resposta"])